# Pedestal Measurement and Calorimeter Calibration

In this notebook you will use the August BL4S 2026 logbook and the converted ROOT files to measure QDC pedestals and calibrate the calorimeter channels.

The logbook is the starting point for finding run numbers. The ROOT files are the final check: during commissioning, channel assignments can change and logbook notes can be incomplete, so each run number and detector channel should be verified with data.

Please don't forget to check our logbook generally, both for August and June.  
**warning**
There are paths according to Berare's computer, you'd need to update them in order to run it on your workspace!





## Goals

1. Find the pedestal runs in the logbook.
2. Check that the converted ROOT files exist.
3. Measure pedestals for all QDC channels.
4. Compare HV-on and HV-off pedestal runs.
5. Use the updated mapping where **CAL0 is on QDC channel 16**.
6. Find calibration runs for the calorimeters.
7. Measure calibration peak positions and fit QDC response versus beam energy.
8. Use the CAL5/CAL1 case to separate an electronics/QDC issue from a beam-position issue.


In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import ROOT
except Exception as exc:
    raise RuntimeError(
        "PyROOT is required for this notebook. Make sure ROOT is set up before starting Jupyter."
    ) from exc

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["figure.dpi"] = 110

# Data files can live in the shared TDAQ_stateOfArt directory, or in a local
# convertedToROOT folder next to this notebook. The absolute path is tried first
# because that is where the August calibration/pedestal ROOT files are stored.
DATA_DIRS = [
    Path("/Users/berare/BL4S/TDAQ_stateOfArt"),
    Path("convertedToROOT"),
    Path("../convertedToROOT"),
]
DATA_DIR = next((p for p in DATA_DIRS if p.exists()), DATA_DIRS[0])
print("Using data directory:", DATA_DIR.resolve())


## Detector to QDC Channel Map

The VME/QDC map labels the calorimeter channels as `QDC0_ch0`, `QDC0_ch1`, etc. During this commissioning period, **CAL0 was moved to channel 16** because channel 0 was problematic.

Use this channel map for the pedestal and calibration measurements below.


In [ ]:
# Final working map for this notebook.
# CAL0 is intentionally channel 16.
CAL_QDC_CHANNEL = {
    "CAL0": 16,
    "CAL1": 1,
    "CAL2": 2,
    "CAL4": 3,
    "CAL5": 4,
    "CAL7": 5,
    "CAL8": 6,
    "CAL9": 7,
    "CAL10": 8,
    "CAL11": 9,
    "CAL12": 10,
    "CAL13": 11,
    "CAL14": 12,
    "CAL17": 13,
    "CAL18": 14,
    "CAL19": 15,
}

ALL_QDC_CHANNELS = list(range(32))
CAL_QDC_CHANNEL


## Helper Functions

These functions open ROOT files, read QDC branches, measure pedestals, and estimate peak positions.


In [ ]:
def root_path(run):
    run = str(run)
    return DATA_DIR / f"{run}.root"


def file_exists(run):
    return root_path(run).exists()


def open_tree(run, tree_name="RAWdata"):
    path = root_path(run)
    if not path.exists():
        raise FileNotFoundError(f"Missing ROOT file: {path}")
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie():
        raise OSError(f"Could not open ROOT file: {path}")
    tree = f.Get(tree_name)
    if not tree:
        raise KeyError(f"Tree {tree_name!r} not found in {path}")
    return f, tree


def qdc_values(run, channel, max_events=None):
    f, tree = open_tree(run)
    branch = f"QDC0_ch{channel}"
    n = tree.GetEntries()
    if max_events is not None:
        n = min(n, int(max_events))
    values = np.empty(n, dtype=float)
    for i in range(n):
        tree.GetEntry(i)
        values[i] = getattr(tree, branch)
    f.Close()
    return values


def robust_mean_std(values, low_percentile=1, high_percentile=99):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lo, hi = np.percentile(values, [low_percentile, high_percentile])
    core = values[(values >= lo) & (values <= hi)]
    return float(np.mean(core)), float(np.std(core, ddof=1))


def estimate_peak(values, bins=240, window_bins=3):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lo, hi = np.percentile(values, [0.5, 99.5])
    hist, edges = np.histogram(values, bins=bins, range=(lo, hi))
    i_max = int(np.argmax(hist))
    left = edges[max(0, i_max - window_bins)]
    right = edges[min(len(edges) - 1, i_max + window_bins + 1)]
    peak_values = values[(values >= left) & (values <= right)]
    if len(peak_values) < 10:
        peak_values = values
    return float(np.mean(peak_values)), float(np.std(peak_values, ddof=1))



def window_estimate(values, center, half_width=180):
    """Estimate the signal peak mean inside a window around an expected center."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    in_window = values[(values >= center - half_width) & (values <= center + half_width)]
    if len(in_window) < 10:
        return estimate_peak(values)
    return float(np.mean(in_window)), float(np.std(in_window, ddof=1))


def plot_qdc(run, channels, labels=None, bins=180, xlim=None, title=None):
    if isinstance(channels, int):
        channels = [channels]
    if labels is None:
        labels = [f"QDC0_ch{ch}" for ch in channels]
    for ch, label in zip(channels, labels):
        vals = qdc_values(run, ch)
        plt.hist(vals, bins=bins, histtype="step", linewidth=1.8, label=label)
    if xlim:
        plt.xlim(*xlim)
    plt.yscale("log")
    plt.xlabel("QDC counts")
    plt.ylabel("entries")
    if title:
        plt.title(title)
    plt.legend()
    plt.show()


## Part 1 - Find the Pedestal Runs

Look in the August logbook around the DWC/pedestal commissioning notes. Fill in the two pedestal run numbers below.

These pedestal runs read out **all QDC channels**, including channel 16. Since CAL0 was later moved to `QDC0_ch16`, no separate pedestal run is needed for channel 16; it is already included in the same all-channel pedestal runs.


In [ ]:
# TODO: fill these from the logbook before running the checkpoint.
student_pedestal_runs = {
    "hv_on_old_cal0_channel": None,
    "hv_off_old_cal0_channel": None,
}
student_pedestal_runs


In [ ]:
# Checkpoint: run after filling student_pedestal_runs.
EXPECTED_PEDESTAL_RUNS = {
    "hv_on_old_cal0_channel": "1786828994",
    "hv_off_old_cal0_channel": "1786829188",
}

for key, expected in EXPECTED_PEDESTAL_RUNS.items():
    got = student_pedestal_runs.get(key)
    status = "OK" if (str(got) if got is not None else None) == expected else "CHECK"
    print(f"{key:26s} your value = {got!s:>12s} expected = {expected!s:>12s}  {status}")


In [ ]:
# File checkpoint.
# If these are missing, upload/convert the raw files into convertedToROOT/ first.
for key, run in EXPECTED_PEDESTAL_RUNS.items():
    if run is None:
        print(f"{key:26s}: no run number in logbook")
    else:
        print(f"{key:26s}: {run} exists = {file_exists(run)}  path = {root_path(run)}")


## Part 2 - Measure Pedestals for All QDC Channels

A pedestal run measures the electronics baseline. We measure it for every QDC channel, not only the calorimeter channels, because bad QDC channels can mimic detector signals.

We compare HV-on and HV-off. For a pure electronics pedestal, turning HV on/off should not produce a large shift. A large difference can indicate detector noise, dark current, pickup, or a cabling/electronics problem.


In [ ]:
def measure_pedestals(run, channels=ALL_QDC_CHANNELS):
    rows = []
    for ch in channels:
        vals = qdc_values(run, ch)
        mean, std = robust_mean_std(vals)
        rows.append({"channel": ch, "mean": mean, "std": std, "n": len(vals)})
    return rows


def print_pedestal_table(rows, title):
    print(title)
    print("channel   mean      std       entries")
    for row in rows:
        print(f"{row['channel']:7d} {row['mean']:8.2f} {row['std']:8.2f} {row['n']:10d}")

# Example once files are present:
# ped_hv_on = measure_pedestals(EXPECTED_PEDESTAL_RUNS["hv_on_old_cal0_channel"])
# ped_hv_off = measure_pedestals(EXPECTED_PEDESTAL_RUNS["hv_off_old_cal0_channel"])
# print_pedestal_table(ped_hv_on, "HV on pedestals")
# print_pedestal_table(ped_hv_off, "HV off pedestals")


In [ ]:
def compare_pedestal_runs(run_a, run_b, label_a="HV on", label_b="HV off", channels=ALL_QDC_CHANNELS):
    a = measure_pedestals(run_a, channels)
    b = measure_pedestals(run_b, channels)
    by_ch_a = {row["channel"]: row for row in a}
    by_ch_b = {row["channel"]: row for row in b}

    print(f"Comparing {label_a} ({run_a}) and {label_b} ({run_b})")
    print("channel   mean_a   mean_b   delta   std_a   std_b")
    deltas = []
    for ch in channels:
        ra, rb = by_ch_a[ch], by_ch_b[ch]
        delta = ra["mean"] - rb["mean"]
        deltas.append(delta)
        print(f"{ch:7d} {ra['mean']:8.2f} {rb['mean']:8.2f} {delta:7.2f} {ra['std']:7.2f} {rb['std']:7.2f}")

    plt.axhline(0, color="black", linewidth=1)
    plt.bar(list(channels), deltas, color="steelblue")
    plt.xlabel("QDC channel")
    plt.ylabel(f"pedestal mean difference: {label_a} - {label_b}")
    plt.title("Pedestal shift from HV-on to HV-off")
    plt.show()

# Example once files are present:
# compare_pedestal_runs("1786828994", "1786829188")


### Pedestal Reference Values

After you measure the pedestals, compare your values with the reference values below. They were computed from the ROOT files in `/Users/berare/BL4S/TDAQ_stateOfArt` using the same trimmed-mean method as `measure_pedestals`.

Only the HV-on and HV-off pedestal runs are used. Both runs contain all QDC channels, so the reference for `QDC0_ch16` is already included.


In [ ]:
REFERENCE_PEDESTALS = {'hv_off_old_cal0_channel': {0: {'mean': 2209.128, 'n': 101572, 'run': '1786829188', 'std': 1645.376},
                             1: {'mean': 107.537, 'n': 101572, 'run': '1786829188', 'std': 2.687},
                             2: {'mean': 120.744, 'n': 101572, 'run': '1786829188', 'std': 4.548},
                             3: {'mean': 116.15, 'n': 101572, 'run': '1786829188', 'std': 0.954},
                             4: {'mean': 95.673, 'n': 101572, 'run': '1786829188', 'std': 1.407},
                             5: {'mean': 120.202, 'n': 101572, 'run': '1786829188', 'std': 0.934},
                             6: {'mean': 104.303, 'n': 101572, 'run': '1786829188', 'std': 1.146},
                             7: {'mean': 101.456, 'n': 101572, 'run': '1786829188', 'std': 1.455},
                             8: {'mean': 106.423, 'n': 101572, 'run': '1786829188', 'std': 1.068},
                             9: {'mean': 135.772, 'n': 101572, 'run': '1786829188', 'std': 1.265},
                             10: {'mean': 107.206, 'n': 101572, 'run': '1786829188', 'std': 0.904},
                             11: {'mean': 108.267, 'n': 101572, 'run': '1786829188', 'std': 1.593},
                             12: {'mean': 105.008, 'n': 101572, 'run': '1786829188', 'std': 0.895},
                             13: {'mean': 73.713, 'n': 101572, 'run': '1786829188', 'std': 1.049},
                             14: {'mean': 89.185, 'n': 101572, 'run': '1786829188', 'std': 1.359},
                             15: {'mean': 125.346, 'n': 101572, 'run': '1786829188', 'std': 5.805},
                             16: {'mean': 121.129, 'n': 101572, 'run': '1786829188', 'std': 0.553},
                             17: {'mean': 98.854, 'n': 101572, 'run': '1786829188', 'std': 0.582},
                             18: {'mean': 126.71, 'n': 101572, 'run': '1786829188', 'std': 0.512},
                             19: {'mean': 116.537, 'n': 101572, 'run': '1786829188', 'std': 0.546},
                             20: {'mean': 97.058, 'n': 101572, 'run': '1786829188', 'std': 0.465},
                             21: {'mean': 133.63, 'n': 101572, 'run': '1786829188', 'std': 0.51},
                             22: {'mean': 127.685, 'n': 101572, 'run': '1786829188', 'std': 0.512},
                             23: {'mean': 153.936, 'n': 101572, 'run': '1786829188', 'std': 0.468},
                             24: {'mean': 110.496, 'n': 101572, 'run': '1786829188', 'std': 0.5},
                             25: {'mean': 96.675, 'n': 101572, 'run': '1786829188', 'std': 0.52},
                             26: {'mean': 114.871, 'n': 101572, 'run': '1786829188', 'std': 0.409},
                             27: {'mean': 128.783, 'n': 101572, 'run': '1786829188', 'std': 0.501},
                             28: {'mean': 95.109, 'n': 101572, 'run': '1786829188', 'std': 0.41},
                             29: {'mean': 96.516, 'n': 101572, 'run': '1786829188', 'std': 0.5},
                             30: {'mean': 85.105, 'n': 101572, 'run': '1786829188', 'std': 0.528},
                             31: {'mean': 100.915, 'n': 101572, 'run': '1786829188', 'std': 0.474}},
 'hv_on_old_cal0_channel': {0: {'mean': 2148.818, 'n': 101648, 'run': '1786828994', 'std': 1613.314},
                            1: {'mean': 107.607, 'n': 101648, 'run': '1786828994', 'std': 3.605},
                            2: {'mean': 120.767, 'n': 101648, 'run': '1786828994', 'std': 6.84},
                            3: {'mean': 116.193, 'n': 101648, 'run': '1786828994', 'std': 1.19},
                            4: {'mean': 95.647, 'n': 101648, 'run': '1786828994', 'std': 1.639},
                            5: {'mean': 120.23, 'n': 101648, 'run': '1786828994', 'std': 1.07},
                            6: {'mean': 104.301, 'n': 101648, 'run': '1786828994', 'std': 1.538},
                            7: {'mean': 101.492, 'n': 101648, 'run': '1786828994', 'std': 1.744},
                            8: {'mean': 106.458, 'n': 101648, 'run': '1786828994', 'std': 1.348},
                            9: {'mean': 135.756, 'n': 101648, 'run': '1786828994', 'std': 1.397},
                            10: {'mean': 107.232, 'n': 101648, 'run': '1786828994', 'std': 1.032},
                            11: {'mean': 108.35, 'n': 101648, 'run': '1786828994', 'std': 2.357},
                            12: {'mean': 105.005, 'n': 101648, 'run': '1786828994', 'std': 0.992},
                            13: {'mean': 73.763, 'n': 101648, 'run': '1786828994', 'std': 1.286},
                            14: {'mean': 89.174, 'n': 101648, 'run': '1786828994', 'std': 2.664},
                            15: {'mean': 125.381, 'n': 101648, 'run': '1786828994', 'std': 10.798},
                            16: {'mean': 121.124, 'n': 101648, 'run': '1786828994', 'std': 0.562},
                            17: {'mean': 98.867, 'n': 101648, 'run': '1786828994', 'std': 0.577},
                            18: {'mean': 126.697, 'n': 101648, 'run': '1786828994', 'std': 0.513},
                            19: {'mean': 116.567, 'n': 101648, 'run': '1786828994', 'std': 0.54},
                            20: {'mean': 97.074, 'n': 101648, 'run': '1786828994', 'std': 0.471},
                            21: {'mean': 133.671, 'n': 101648, 'run': '1786828994', 'std': 0.503},
                            22: {'mean': 127.68, 'n': 101648, 'run': '1786828994', 'std': 0.511},
                            23: {'mean': 153.962, 'n': 101648, 'run': '1786828994', 'std': 0.461},
                            24: {'mean': 110.499, 'n': 101648, 'run': '1786828994', 'std': 0.5},
                            25: {'mean': 96.691, 'n': 101648, 'run': '1786828994', 'std': 0.519},
                            26: {'mean': 114.868, 'n': 101648, 'run': '1786828994', 'std': 0.411},
                            27: {'mean': 128.791, 'n': 101648, 'run': '1786828994', 'std': 0.504},
                            28: {'mean': 95.112, 'n': 101648, 'run': '1786828994', 'std': 0.414},
                            29: {'mean': 96.523, 'n': 101648, 'run': '1786828994', 'std': 0.499},
                            30: {'mean': 85.11, 'n': 101648, 'run': '1786828994', 'std': 0.53},
                            31: {'mean': 100.927, 'n': 101648, 'run': '1786828994', 'std': 0.472}}}


def rows_to_mean_dict(rows):
    return {row["channel"]: row["mean"] for row in rows}


def compare_pedestal_reference(label, student_means, tolerance=2.0):
    """Compare a student pedestal mean dictionary to the reference table."""
    ref = REFERENCE_PEDESTALS[label]
    print(f"Comparing pedestal label: {label}")
    print("channel   student   reference   diff   status")
    for ch in sorted(ref):
        if ch not in student_means:
            print(f"{ch:7d}   MISSING   {ref[ch]['mean']:9.3f}      --   CHECK")
            continue
        diff = student_means[ch] - ref[ch]["mean"]
        status = "OK" if abs(diff) <= tolerance else "CHECK"
        print(f"{ch:7d} {student_means[ch]:9.3f} {ref[ch]['mean']:9.3f} {diff:7.3f}   {status}")

# Example student workflow:
# ped_hv_off = measure_pedestals("1786829188")
# compare_pedestal_reference("hv_off_old_cal0_channel", rows_to_mean_dict(ped_hv_off))


### CAL0 Channel 16 Check

The logbook says CAL0 was moved to channel 16. Because the pedestal runs include all QDC channels, `QDC0_ch16` already has pedestal values in both the HV-on and HV-off pedestal runs.

Use the plot below to compare the old problematic channel 0 with channel 16 in the same pedestal runs.


In [ ]:
# Example:
# plot_qdc("1786829188", [0, 16], labels=["old ch0", "CAL0 ch16"], title="CAL0 channel 16 pedestal check")


## Part 3 - Find Calibration Runs

Now use the logbook to find calibration runs. Each detector should have up to three beam energies: -1, -2, and -3 GeV.

Fill your own run list first. Then use the checkpoint dictionary below to check the run numbers and discuss any differences between the logbook notes and the data.


In [ ]:
# TODO: fill at least a few detectors yourself from the logbook before running the checkpoint.
# Format: "CALX": {-1: "run_for_minus_1_GeV", -2: "run_for_minus_2_GeV", -3: "run_for_minus_3_GeV"}
student_calibration_runs = {
    "CAL0": {-1: None, -2: None, -3: None},
    "CAL1": {-1: None, -2: None, -3: None},
    "CAL5": {-1: None, -2: None, -3: None},
}
student_calibration_runs


In [ ]:
# Checkpoint values from the August logbook.
# Some early runs were repeated after channel/electronics issues; this table uses the later/cleaner entries where available.
EXPECTED_CALIBRATION_RUNS = {
    "CAL0": {-1: "1786835122", -2: "1786834513", -3: "1786833136"},  # CAL0 moved to QDC ch16
    "CAL1": {-1: "1786800039", -2: "1786799056", -3: "1786798201"},
    "CAL2": {-1: "1786781408", -2: "1786780617", -3: "1786779710"},
    "CAL4": {-1: "1786870373", -2: "1786869626", -3: "1786868952"},
    "CAL5": {-1: "1786865850", -2: "1786866862", -3: "1786867665"},
    "CAL7": {-1: "1786836376", -2: "1786837396", -3: "1786838309"},
    "CAL8": {-1: "1786714515", -2: "1786713224", -3: "1786711761"},
    "CAL9": {-1: "1786741940", -2: "1786739830", -3: "1786726168"},
    "CAL10": {-1: "1786793439", -2: "1786791937", -3: "1786790499"},
    "CAL11": {-1: "1786747305", -2: "1786746501", -3: "1786745756"},
    "CAL12": {-1: "1786801341", -2: "1786802590", -3: "1786803907"},
    "CAL13": {-1: "1786743090", -2: "1786744038", -3: "1786744833"},
    "CAL14": {-1: "1786748346", -2: "1786750147", -3: "1786750887"},
    "CAL17": {-1: "1786807867", -2: "1786806354", -3: "1786805198"},
    "CAL18": {-1: "1786864539", -2: "1786840283", -3: "1786839384"},
    "CAL19": {-1: "1786723178", -2: "1786724681", -3: "1786725412"},
}

for det, runs in student_calibration_runs.items():
    expected = EXPECTED_CALIBRATION_RUNS.get(det, {})
    for energy, got in runs.items():
        exp = expected.get(energy)
        status = "OK" if (str(got) if got is not None else None) == exp else "CHECK"
        print(f"{det:5s} {energy:2d} GeV   your value = {got!s:>12s} expected = {exp!s:>12s}  {status}")


In [ ]:
# File checkpoint for all expected calibration runs.
missing = []
for det, runs in EXPECTED_CALIBRATION_RUNS.items():
    for energy, run in runs.items():
        if run is None:
            continue
        ok = file_exists(run)
        if not ok:
            missing.append(run)
        print(f"{det:5s} {energy:2d} GeV run {run}: exists = {ok}")

print()
print("Missing files:", sorted(set(missing)))


## Part 4 - Measure Calibration Peaks

For each detector and energy:

1. Read the detector's QDC channel.
2. Subtract the pedestal for that channel if a pedestal table is available.
3. Estimate the peak position.
4. Fit peak position versus beam energy.

The logbook values below are approximate visual centers used to choose a reasonable signal-peak window. They are checkpoints, not a replacement for measuring the ROOT files.

**CAL8 note:** use the CAL8 calibration set `1786714515`, `1786713224`, `1786711761`. The `-1 GeV` logbook observation has `****`; this means the centered QDC value was not written down, not that the run should be excluded. In this notebook the `-1 GeV` signal peak is measured directly from the ROOT file. The earlier `1786708920` run is useful for discussion because the logbook flags it as suspicious.


In [ ]:
# Approximate QDC signal-peak centers written in the logbook.
# These define the search windows used for the reference calibration values.
LOGBOOK_QDC_CENTERS = {
    "CAL0": {-1: 1000, -2: 1900, -3: 2800},
    "CAL1": {-1: 820, -2: 1516, -3: 2308},
    "CAL2": {-1: 1200, -2: 2200, -3: 3200},
    "CAL4": {-1: 1112, -2: 2107, -3: 3119},
    "CAL5": {-1: 697, -2: 1600, -3: 3015},
    "CAL7": {-1: 1100, -2: 2100, -3: 3000},
    "CAL8": {-1: 1100, -2: 1681, -3: 2600},
    "CAL9": {-1: 1200, -2: 2300, -3: 3500},
    "CAL10": {-1: 1000, -2: 1700, -3: 2400},
    "CAL11": {-1: 900, -2: 1800, -3: 2600},
    "CAL12": {-1: 750, -2: 1400, -3: 2000},
    "CAL13": {-1: 900, -2: 1600, -3: 2400},
    "CAL14": {-1: 1200, -2: 2300, -3: 3300},
    "CAL17": {-1: 1000, -2: 1900, -3: 2700},
    "CAL18": {-1: 1000, -2: 1900, -3: 2800},
    "CAL19": {-1: 800, -2: 1200, -3: 1700},
}

# Some calibration files are very large. The checkpoint calculation can use a
# limited sample while keeping the same channel and peak-finding method.
CALIBRATION_MAX_EVENTS = {
    ("CAL18", -2): 100_000,
}

LOGBOOK_QDC_CENTERS


In [ ]:
def measure_detector_calibration(detector, pedestal_by_channel=None, half_width=180):
    ch = CAL_QDC_CHANNEL[detector]
    rows = []
    for energy, run in EXPECTED_CALIBRATION_RUNS[detector].items():
        if run is None or not file_exists(run):
            continue
        max_events = CALIBRATION_MAX_EVENTS.get((detector, energy))
        vals = qdc_values(run, ch, max_events=max_events)
        pedestal = 0.0
        if pedestal_by_channel is not None and ch in pedestal_by_channel:
            pedestal = pedestal_by_channel[ch]
        vals = vals - pedestal
        expected_center = LOGBOOK_QDC_CENTERS.get(detector, {}).get(energy)
        if expected_center is None:
            peak, width = estimate_peak(vals)
        else:
            peak, width = window_estimate(vals, expected_center, half_width=half_width)
        rows.append({"detector": detector, "energy_GeV": abs(energy), "run": run, "channel": ch, "peak": peak, "width": width})
    return rows


def fit_calibration(rows):
    if len(rows) < 2:
        return None
    x = np.array([r["energy_GeV"] for r in rows], dtype=float)
    y = np.array([r["peak"] for r in rows], dtype=float)
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope), float(intercept)


def plot_calibration(detector, rows, expected_centers=None):
    if not rows:
        print(f"No ROOT calibration rows for {detector}")
        return
    x = np.array([r["energy_GeV"] for r in rows], dtype=float)
    y = np.array([r["peak"] for r in rows], dtype=float)
    plt.scatter(x, y, s=70, label="measured from ROOT")

    fit = fit_calibration(rows)
    if fit:
        slope, intercept = fit
        xx = np.linspace(min(x) * 0.9, max(x) * 1.05, 100)
        plt.plot(xx, slope * xx + intercept, label=f"fit: QDC = {slope:.1f} E + {intercept:.1f}")

    if expected_centers:
        ex = []
        ey = []
        for energy, center in expected_centers.items():
            if center is not None:
                ex.append(abs(energy))
                ey.append(center)
        if ex:
            plt.scatter(ex, ey, marker="x", s=80, label="logbook approx")

    plt.xlabel("beam energy magnitude [GeV]")
    plt.ylabel("QDC signal-peak position")
    plt.title(f"{detector} calibration")
    plt.legend()
    plt.show()

# Example:
# rows = measure_detector_calibration("CAL0")
# plot_calibration("CAL0", rows, LOGBOOK_QDC_CENTERS["CAL0"])
# fit_calibration(rows)


### Calibration Reference Values

These are the reference signal-peak positions and linear calibration fits computed from the ROOT files. Run your own measurement first, then compare your peak positions and fit parameters against these values.


In [ ]:
REFERENCE_CALIBRATION_PEAKS = {'CAL0': {-3: {'channel': 16, 'n': 27759, 'peak': 2805.024, 'run': '1786833136', 'width': 98.88},
          -2: {'channel': 16, 'n': 16185, 'peak': 1909.106, 'run': '1786834513', 'width': 95.522},
          -1: {'channel': 16, 'n': 23958, 'peak': 1010.608, 'run': '1786835122', 'width': 87.608}},
 'CAL1': {-3: {'channel': 1, 'n': 27804, 'peak': 2318.121, 'run': '1786798201', 'width': 93.509},
          -2: {'channel': 1, 'n': 29111, 'peak': 1543.962, 'run': '1786799056', 'width': 90.057},
          -1: {'channel': 1, 'n': 26111, 'peak': 813.491, 'run': '1786800039', 'width': 77.837}},
 'CAL10': {-3: {'channel': 8, 'n': 26643, 'peak': 2424.37, 'run': '1786790499', 'width': 98.772},
           -2: {'channel': 8, 'n': 28910, 'peak': 1708.26, 'run': '1786791937', 'width': 95.231},
           -1: {'channel': 8, 'n': 51744, 'peak': 943.266, 'run': '1786793439', 'width': 74.56}},
 'CAL11': {-3: {'channel': 9, 'n': 30630, 'peak': 2603.065, 'run': '1786745756', 'width': 93.555},
           -2: {'channel': 9, 'n': 29886, 'peak': 1782.769, 'run': '1786746501', 'width': 87.413},
           -1: {'channel': 9, 'n': 26695, 'peak': 927.369, 'run': '1786747305', 'width': 83.557}},
 'CAL12': {-3: {'channel': 10, 'n': 22283, 'peak': 2011.775, 'run': '1786803907', 'width': 98.672},
           -2: {'channel': 10, 'n': 27116, 'peak': 1405.053, 'run': '1786802590', 'width': 94.535},
           -1: {'channel': 10, 'n': 27977, 'peak': 755.451, 'run': '1786801341', 'width': 85.993}},
 'CAL13': {-3: {'channel': 11, 'n': 31310, 'peak': 2402.591, 'run': '1786744833', 'width': 96.97},
           -2: {'channel': 11, 'n': 29658, 'peak': 1618.515, 'run': '1786744038', 'width': 92.165},
           -1: {'channel': 11, 'n': 24579, 'peak': 876.73, 'run': '1786743090', 'width': 78.166}},
 'CAL14': {-3: {'channel': 12, 'n': 1006805, 'peak': 3306.341, 'run': '1786750887', 'width': 98.143},
           -2: {'channel': 12, 'n': 25485, 'peak': 2281.066, 'run': '1786750147', 'width': 92.554},
           -1: {'channel': 12, 'n': 21953, 'peak': 1184.117, 'run': '1786748346', 'width': 86.061}},
 'CAL17': {-3: {'channel': 13, 'n': 22121, 'peak': 2713.474, 'run': '1786805198', 'width': 99.966},
           -2: {'channel': 13, 'n': 25079, 'peak': 1894.949, 'run': '1786806354', 'width': 96.077},
           -1: {'channel': 13, 'n': 27672, 'peak': 992.925, 'run': '1786807867', 'width': 87.297}},
 'CAL18': {-3: {'channel': 14, 'n': 22358, 'peak': 2804.475, 'run': '1786839384', 'width': 99.071},
           -2: {'channel': 14, 'n': 100000, 'peak': 1904.207, 'run': '1786840283', 'width': 96.044},
           -1: {'channel': 14, 'n': 23005, 'peak': 1003.023, 'run': '1786864539', 'width': 87.1}},
 'CAL19': {-3: {'channel': 15, 'n': 30984, 'peak': 1710.031, 'run': '1786725412', 'width': 91.333},
           -2: {'channel': 15, 'n': 33822, 'peak': 1194.641, 'run': '1786724681', 'width': 84.252},
           -1: {'channel': 15, 'n': 18537, 'peak': 694.8, 'run': '1786723178', 'width': 49.079}},
 'CAL2': {-3: {'channel': 2, 'n': 29285, 'peak': 3195.872, 'run': '1786779710', 'width': 96.666},
          -2: {'channel': 2, 'n': 28114, 'peak': 2177.064, 'run': '1786780617', 'width': 91.413},
          -1: {'channel': 2, 'n': 21750, 'peak': 1145.712, 'run': '1786781408', 'width': 76.171}},
 'CAL4': {-3: {'channel': 3, 'n': 26438, 'peak': 3130.855, 'run': '1786868952', 'width': 96.662},
          -2: {'channel': 3, 'n': 28972, 'peak': 2114.902, 'run': '1786869626', 'width': 93.393},
          -1: {'channel': 3, 'n': 22992, 'peak': 1096.921, 'run': '1786870373', 'width': 83.053}},
 'CAL5': {-3: {'channel': 4, 'n': 23055, 'peak': 3036.919, 'run': '1786867665', 'width': 97.467},
          -2: {'channel': 4, 'n': 5064, 'peak': 1630.495, 'run': '1786866862', 'width': 102.262},
          -1: {'channel': 4, 'n': 7314, 'peak': 718.97, 'run': '1786865850', 'width': 106.417}},
 'CAL7': {-3: {'channel': 5, 'n': 24984, 'peak': 3005.458, 'run': '1786838309', 'width': 99.053},
          -2: {'channel': 5, 'n': 24259, 'peak': 2089.787, 'run': '1786837396', 'width': 95.104},
          -1: {'channel': 5, 'n': 18446, 'peak': 1102.096, 'run': '1786836376', 'width': 87.649}},
 'CAL8': {-3: {'channel': 6, 'n': 4047, 'peak': 2642.518, 'run': '1786711761', 'width': 101.122},
          -2: {'channel': 6, 'n': 5720, 'peak': 1726.058, 'run': '1786713224', 'width': 99.944},
          -1: {'channel': 6, 'n': 25862, 'peak': 1088.13, 'run': '1786714515', 'width': 85.946}},
 'CAL9': {-3: {'channel': 7, 'n': 312632, 'peak': 3505.654, 'run': '1786726168', 'width': 97.636},
          -2: {'channel': 7, 'n': 47612, 'peak': 2323.356, 'run': '1786739830', 'width': 94.775},
          -1: {'channel': 7, 'n': 23216, 'peak': 1202.939, 'run': '1786741940', 'width': 88.065}}}

REFERENCE_CALIBRATION_FITS = {'CAL0': {'intercept': 113.83, 'n_points': 3, 'slope': 897.208},
 'CAL1': {'intercept': 53.895, 'n_points': 3, 'slope': 752.315},
 'CAL10': {'intercept': 210.862, 'n_points': 3, 'slope': 740.552},
 'CAL11': {'intercept': 95.372, 'n_points': 3, 'slope': 837.848},
 'CAL12': {'intercept': 134.436, 'n_points': 3, 'slope': 628.162},
 'CAL13': {'intercept': 106.752, 'n_points': 3, 'slope': 762.93},
 'CAL14': {'intercept': 134.95, 'n_points': 3, 'slope': 1061.112},
 'CAL17': {'intercept': 146.567, 'n_points': 3, 'slope': 860.274},
 'CAL18': {'intercept': 102.45, 'n_points': 3, 'slope': 900.726},
 'CAL19': {'intercept': 184.593, 'n_points': 3, 'slope': 507.616},
 'CAL2': {'intercept': 122.722, 'n_points': 3, 'slope': 1025.08},
 'CAL4': {'intercept': 80.293, 'n_points': 3, 'slope': 1016.967},
 'CAL5': {'intercept': -522.488, 'n_points': 3, 'slope': 1158.974},
 'CAL7': {'intercept': 162.419, 'n_points': 3, 'slope': 951.681},
 'CAL8': {'intercept': 264.514, 'n_points': 3, 'slope': 777.194},
 'CAL9': {'intercept': 41.269, 'n_points': 3, 'slope': 1151.357}}

REFERENCE_CALIBRATION_PEAKS, REFERENCE_CALIBRATION_FITS


## Calibration Table

After measuring each detector, enter your fit values below and compare them to the notebook's computed values.


In [ ]:
# TODO: after running the fits, enter your values here.
# Units: slope is QDC counts / GeV; intercept is QDC counts.
student_calibration_constants = {
    # "CAL0": {"slope": None, "intercept": None},
}
student_calibration_constants


In [ ]:
# Compute calibration constants from available ROOT files using the same method as the reference table.
notebook_calibration_constants = {}
for det in EXPECTED_CALIBRATION_RUNS:
    if det not in CAL_QDC_CHANNEL:
        continue
    rows = measure_detector_calibration(det)
    fit = fit_calibration(rows)
    if fit is None:
        continue
    slope, intercept = fit
    notebook_calibration_constants[det] = {"slope": slope, "intercept": intercept, "n_points": len(rows)}

notebook_calibration_constants


In [ ]:
# Check your entered values against the reference values.
for det, yours in student_calibration_constants.items():
    ref = REFERENCE_CALIBRATION_FITS.get(det)
    if ref is None:
        print(f"{det}: no reference available")
        continue
    ds = None if yours.get("slope") is None else yours["slope"] - ref["slope"]
    di = None if yours.get("intercept") is None else yours["intercept"] - ref["intercept"]
    print(f"{det}: slope yours={yours.get('slope')} ref={ref['slope']:.2f} diff={ds}")
    print(f"     intercept yours={yours.get('intercept')} ref={ref['intercept']:.2f} diff={di}")


## CAL5 / CAL1 / QDC Fault Check

The logbook notes that CAL5 was repeated because neighboring ECALs showed high QDC counts. Later notes point to an awkward CAL1 signal and unusually high readings on QDC channels 0 and 1.

This section is a diagnostic check. Use the plots to decide whether the high neighboring signal is caused by the beam position or by the QDC/electronics chain.

If the beam is centered on CAL5 but CAL1/QDC ch1 is high, the likely explanation is an electronics/QDC issue around CAL1/QDC ch1, not a wrong center position for CAL5.


In [ ]:
# Draw CAL5 calibration run together with CAL1 and nearby channels.
# This helps show whether an apparent neighbor signal is an electronics/QDC issue.
CAL5_PROBLEM_RUNS = ["1786865850", "1786866862", "1786867665"]
channels_to_compare = [CAL_QDC_CHANNEL["CAL1"], CAL_QDC_CHANNEL["CAL5"], CAL_QDC_CHANNEL["CAL4"], CAL_QDC_CHANNEL["CAL7"]]
labels_to_compare = ["CAL1 / QDC ch1", "CAL5 / QDC ch4", "CAL4 / QDC ch3", "CAL7 / QDC ch5"]

# Example once files are present:
# for run in CAL5_PROBLEM_RUNS:
#     if file_exists(run):
#         plot_qdc(run, channels_to_compare, labels_to_compare, title=f"CAL5/CAL1 electronics check, run {run}")


### Interpretation Check

When you look at the CAL5 comparison plot, answer these questions:

1. Which QDC channel has the suspicious high signal?
2. Is the suspicious signal consistent across energies or runs?
3. Does the beam-position information say the beam was centered on CAL5?
4. If the beam was centered but CAL1/QDC ch1 is high, what is the most likely explanation?

Expected conclusion: this was an electronics/QDC fault around CAL1/QDC ch1, not a problem with the DESY table center position for CAL5.


## Final Checklist

Before using calibration constants in later notebooks:

- The ROOT files are read from `/Users/berare/BL4S/TDAQ_stateOfArt`.
- Pedestals were measured for all 32 QDC channels using runs `1786828994` and `1786829188`.
- Your pedestal means were compared with `REFERENCE_PEDESTALS`.
- HV-on and HV-off pedestals were compared.
- CAL0 uses `QDC0_ch16` after the channel move, and channel 16 is already included in the pedestal runs.
- Calibration runs were checked against `EXPECTED_CALIBRATION_RUNS`.
- Signal-peak positions were compared with `REFERENCE_CALIBRATION_PEAKS`.
- Linear calibration fits were compared with `REFERENCE_CALIBRATION_FITS`.
- CAL5/CAL1/QDC fault behavior was checked with plots, not assumed from the logbook text alone.
